# B：DCT2 局部身份、M3-151 时间候选、CDC32 条件聚合

本交付使用已采用共同r1/c0与B1/B2/B3组件。默认Run all从GitHub的dev/local-joint-state-payload-v1研究分支加载当前可用源码，无需填写源码目录。也可在第二代码cell填写普通可编辑源码包目录。此notebook不会用旧runner替代缺失的新实现。

固定4新来源（2cal/2eval）、OFF/P0/PM三臂、三fixture。DCT2 k25..49，原176状态在解码前最终FP32 latent写入；完整native50、CFG5不变。12 D181、432 E41、codec各48；M3/E2共享全部观察；B3没有新增物理调用。先用新链90个完整null族一次冻结tau，再进入eval。校准缺失不缩分母或回退旧tau。

共同局部C不被路径反剪；R0/R1分别记录；CDC使用全部计划副本，冲突和技术缺失保留。每个两片段bundle独立冻结匿名计划后读缓存；SEP与条件JOIN并列，不认证同源。Rc仅诊断。净32位、三层支撑和失败分母保留。

本轮真实模型/GPU/VAE/媒体工作由用户运行。首代码cell独立挂载Drive。存在任何执行残留时，后续Run all仅读结果；失败/中断消耗计划单位，不自动续算、换源或重试。不要求GPU型号、源码摘要、精确依赖版本或Git状态。

保存盲计划与决定并回读后才开放真值与质量后评。两eval来源及d=0/8固定协议不能证明一般编辑、总体FPR、源认证、新旧载体优劣或原生成期时间控制承诺。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

from pathlib import Path
import json, subprocess, sys, tempfile, urllib.request, zipfile
OUTPUT = Path('/content/drive/MyDrive/Video-WM/B-Terminal-Three-Group-V1/fixed-four-new-v1')
SOURCE_REF = 'dev/local-joint-state-payload-v1'
# Optional editable source folder from the ordinary ZIP. None downloads GitHub.
SOURCE_DIRECTORY = None

def execution_artifacts(output):
    output = Path(output)
    names = ('attempt.json', 'result.json', 'ledger.json', 'protocol.json', 'thresholds.json',
             'planned_outputs.json', 'path_domain.json', 'background.json', 'blind_complete.json', 'evaluation.json')
    found = [n+s for n in names for s in ('', '.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for name in ('private', 'receipts', 'media', 'observations', 'families', 'blind', 'bundles', 'quality'):
        folder = output/name
        if folder.is_dir():
            found.extend(str(p.relative_to(output)) for p in folder.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)

def recover_ledger(output):
    """Read-only receipt overlay; also embedded verbatim in the notebook.

    STARTED remains spent and incomplete. Invalid or uncommitted receipts
    remain unresolved, never an inferred successful or unentered call.
    """
    from collections import Counter
    import json
    from pathlib import Path
    output = Path(output)
    ledger, errors, receipt_slots = {}, {}, []
    try:
        ledger = json.loads((output/'ledger.json').read_text(encoding='utf-8'))
        if not isinstance(ledger, dict) or any(not isinstance(row, dict) for row in ledger.values()):
            raise ValueError('ledger must map slots to records')
    except (OSError, ValueError) as error:
        ledger = {}
        errors['ledger.json'] = str(error)
    directory = output/'receipts'
    paths = sorted(directory.rglob('*.json')) + sorted(directory.rglob('*.json.tmp'))
    for path in paths:
        relative = path.relative_to(directory).as_posix()
        pending = relative.endswith('.tmp')
        slot = relative[:-9] if pending else relative[:-5]
        previous = ledger.get(slot, {})
        try:
            receipt = json.loads(path.read_text(encoding='utf-8'))
            if pending:
                raise ValueError('uncommitted receipt; completion is unresolved')
            if not isinstance(receipt, dict) or receipt.get('slot') != slot:
                raise ValueError('receipt slot does not match its address')
            status, attempts = receipt.get('status'), receipt.get('attempts')
            if status not in ('NOT_ENTERED', 'STARTED', 'RETURNED', 'FAILED', 'INTERRUPTED'):
                raise ValueError('invalid receipt status')
            if type(attempts) is not int or attempts != (0 if status == 'NOT_ENTERED' else 1):
                raise ValueError('invalid receipt attempt count')
            if not isinstance(receipt.get('kind'), str) or not receipt['kind']:
                raise ValueError('receipt kind required')
            if previous.get('kind') is not None and previous['kind'] != receipt['kind']:
                raise ValueError('receipt kind disagrees with saved slot')
            if previous.get('attempts') == 1 and attempts == 0:
                raise ValueError('receipt cannot erase a spent call')
            ledger[slot] = receipt
            receipt_slots.append(slot)
        except (OSError, ValueError) as error:
            name = 'receipts/'+relative
            errors[name] = str(error)
            ledger[slot] = dict(previous, slot=slot, status='UNRESOLVED_RECEIPT',
                                attempts=1 if previous.get('attempts') == 1 else None,
                                saved_status=previous.get('status'), recovery_error=name)
    counts = dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()))
    kinds = sorted({row.get('kind', 'UNKNOWN') for row in ledger.values()})
    unit_counts = {kind: dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()
                                     if row.get('kind', 'UNKNOWN') == kind)) for kind in kinds}
    return dict(ledger=ledger, counts=counts, unit_counts=unit_counts,
                receipt_slots=receipt_slots, read_errors=errors, physical_reentry=False)

EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if EXISTING_ARTIFACTS:
    print('Existing attempt: read saved results only. No setup or scientific calls.')
else:
    OUTPUT.mkdir(parents=True, exist_ok=True)
    if SOURCE_DIRECTORY is None:
        workspace = Path(tempfile.mkdtemp(prefix='SC-SSTW-b-terminal-three-group-v1-', dir='/content'))
        archive = workspace/'source.zip'
        url = 'https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/'+SOURCE_REF
        urllib.request.urlretrieve(url, archive)
        destination = (workspace/'source').resolve()
        destination.mkdir()
        with zipfile.ZipFile(archive) as source:
            for member in source.infolist():
                target = (destination/member.filename).resolve()
                if not target.is_relative_to(destination) or (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('unsafe source archive entry: '+member.filename)
            source.extractall(destination)
        entries = list(destination.glob('*/experiments/wan_state_clock/b_terminal_three_group_v1_run.py'))
        if len(entries) != 1:
            raise FileNotFoundError('New B runner unavailable in downloaded source: '+url)
        SOURCE_ROOT = entries[0].parents[2]
    else:
        SOURCE_ROOT = Path(SOURCE_DIRECTORY).expanduser().resolve()
    required = ['experiments/wan_state_clock/b_terminal_three_group_v1_run.py',
                'main/tube_state/b_local_identity_dct_v1.py', 'main/tube_state/b_m3_time_path_v1.py',
                'main/tube_state/b_cdc32_aggregation_v1.py', 'runtime/wan/b_cdc32_bundle_v1.py']
    for relative in required:
        if not (SOURCE_ROOT/relative).is_file():
            raise FileNotFoundError('New B source file unavailable: '+str(SOURCE_ROOT/relative))
    print('Actual editable source folder:', SOURCE_ROOT)


In [ ]:

if not EXISTING_ARTIFACTS:
    import time
    # Reuse the successfully run Card102 setup, including its torchao repair.
    # Keep this self-contained; no legacy notebook/builder runtime dependency.
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import WanPipeline,AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan pipeline and VAE')
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','-qq','update'], 'Prepare media dependency')
        command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))
    # Probe the actual new imports in a fresh process rooted at this source.
    import_probe = "from pathlib import Path; import importlib; root=Path.cwd().resolve(); names=['runtime.wan.b_terminal_three_group_v1','runtime.wan.b_terminal_three_group_backend_v1','main.tube_state.b_local_identity_dct_v1','main.tube_state.b_m3_time_path_v1','main.tube_state.b_cdc32_aggregation_v1','runtime.wan.b_cdc32_bundle_v1']; modules=[importlib.import_module(n) for n in names]; assert all(Path(m.__file__).resolve().is_relative_to(root) for m in modules); print(*[m.__file__ for m in modules],sep='\\n')"
    subprocess.run([sys.executable, '-c', import_probe], cwd=SOURCE_ROOT, check=True)
    with (OUTPUT/'runner.log').open('w') as stream:
        worker = subprocess.Popen([sys.executable, '-u', '-m',
            'experiments.wan_state_clock.b_terminal_three_group_v1_run', '--output', str(OUTPUT)],
            cwd=SOURCE_ROOT, stdout=stream, stderr=subprocess.STDOUT)
    try:
        while worker.poll() is None:
            time.sleep(30)
            try:
                value = json.loads((OUTPUT/'progress.json').read_text())
                print(value.get('active'), value.get('counts'), flush=True)
            except (OSError, ValueError):
                print('Same worker active; progress record not available yet.', flush=True)
    except KeyboardInterrupt:
        worker.terminate()
        try: worker.wait(timeout=30)
        except subprocess.TimeoutExpired: worker.kill(); worker.wait()
        print('Stopped. Attempt remains spent; later Run all reads saved records.')
        raise
    print('Runner exit:', worker.returncode, 'Saved output:', OUTPUT)


In [ ]:

recovery = recover_ledger(OUTPUT)
print('Receipt-recovered execution counts', recovery['counts'])
print('Receipt-recovered unit counts', recovery['unit_counts'])
for name, error in recovery['read_errors'].items():
    print('Recovery read failure:', name, error)
print('STARTED is spent and incomplete; UNRESOLVED_RECEIPT retains uncertain accounting. No reentry.')
for name in ('result.json', 'thresholds.json', 'evaluation.json', 'planned_outputs.json'):
    path = OUTPUT/name
    if not path.is_file():
        print('Unavailable:', name); continue
    try: value = json.loads(path.read_text())
    except (OSError, ValueError) as error:
        print('Read failure:', name, str(error)); continue
    if name == 'result.json': print('Saved result snapshot', value.get('status'), value.get('denominators'))
    elif name == 'thresholds.json': print('New DCT2-chain tau', value.get('status'), value.get('tau'), 'planned null families', value.get('planned_families'))
    elif name == 'evaluation.json':
        print('Reader totals', value.get('totals'))
        for label in ('media_P0_PM_pair', 'native_P0_PM_pair'):
            print(label, {k:v for k,v in value.get(label,{}).items() if k!='rows'})
        print('Quality pairs', len(value.get('quality',{})), 'conditional bundles',len(value.get('bundles',[])))
        print('Evidence ceiling', value.get('evidence_ceiling'))
    else: print('Fixed planned tasks', len(value.get('tasks',{})), 'bundles', len(value.get('bundles',[])))
print('R0/R1 local bits, paths, common-bit BER, coverage, false attribution and quality remain separate. No scientific or quality PASS.')
print('Failures, NOT_ENTERED, full candidate families and unresolved mappings remain at', OUTPUT)
